# Assembly trigger sweep on Barrett reduction

This notebook finds which instruction of HQC's Barrett reduction leaks. `simpleserial-barrett-asm/barrett.S` holds the same instructions the compiler makes for HQC's code, and sets the trigger pin directly, with one store each: `str r7, [r4]` sets the pin high before the first Barrett instruction, and `str r7, [r6]` sets it low after the instruction that `TRIGGER_STAGE` picks. Each test ends the window one instruction earlier than the test before.

| Test | Trigger low after | What the instruction does |
| ---- | ----------------- | ------------------------- |
| 1 | `str r2, [r5]` | writes the result to `r_out` |
| 2 | `subs r2, r2, r3` | r = r - sel |
| 3 | `ands r3, r1` | sel = mask & N |
| 4 | `negs r3, r3` | mask = -reduce_flag |
| 5 | `lsrs r3, r3, #31` | reduce_flag = (~t) >> 31 |
| 6 | `mvns r3, r3` | ~t |
| 7 | `subs r3, #5` | t = r - N, second step |
| 8 | `sub.w r3, r2, #17664` | t = r - N, first step |
| 9 | `mls r2, r1, r3, r2` | r = x - q * N |
| 10 | `movw r1, #17669` | loads N |
| 11 | `umull r1, r3, r3, r2` | q = (x * mu) >> 32 |
| 12 | `ldr r5, =r_out` | loads the address of `r_out` |
| 13 | `ldr r3, =PARAM_N_MU` | loads mu |
| 14 | `ldr r2, [r5]` | reads x |
| 15 | the pin-high store | nothing: an empty window |

## How to read the tests

- The trigger window holds only the Barrett instructions between the two pin stores. There is no function call, so nothing else runs inside it, and the instructions inside the window are the same in every test. Only the pin-low store moves. The instructions after it sit 2 bytes later than in test 1, which can change their timing a little.
- The plots count clock cycles from trigger high, so trigger high is cycle 0 and Barrett starts there. The vertical line marks trigger low.
- The rest of the reduction runs right after the pin-low store, so its leaks show just after the line.
- Go through the tests in order. If a leaking cycle lies before the line in one test but not in the next, it comes from the instruction that the next test took out.
- Test 15 has an empty window. It is the control: a leak before its line comes from the two pin stores or, within the one-instruction skew below, from `ldr r2, [r5]` reading x.
- The processor works on more than one instruction at a time, and the pin store goes through the peripheral bus, so the edge can be about one instruction off. Read each result as plus or minus one instruction.

In [1]:
import os
import subprocess

import numpy as np
import chipwhisperer as cw
import holoviews as hv
from scipy.stats import ttest_ind
from tqdm.auto import tqdm

N = 17669           # the modulus, must be the same as PARAM_N in barrett.S
FIXED_X = 1212218   # the input used for every fixed-group run
N_WARMUP = 1000     # runs done once before test 1 to warm up the board, their traces are not kept
N_TRACES = 5000     # runs per test, fixed and random together
SAMPLES = 1000      # power samples recorded in each run
SEED = 1            # the same seed gives the same inputs every time
THRESHOLD = 4.5     # the |t| limit, above it in both halves means a leak

FIRMWARE = "simpleserial-barrett-asm"
HEX = FIRMWARE + "/simpleserial-barrett-asm-CWHUSKY.hex"
ELF = FIRMWARE + "/simpleserial-barrett-asm-CWHUSKY.elf"
FIGURES = "figures/asm-sweep"  # folder for the PNG copies of the plots

# Bokeh shows the interactive plots, matplotlib draws the PNG copies
hv.extension("bokeh", "matplotlib", logo=False)

## Setup

Connects to the Husky and the target board, and finds the clock cycle of each sample, counted from trigger high.

In [2]:
scope = cw.scope()
target = cw.target(scope)
scope.default_setup()
scope.adc.samples = SAMPLES

SAMPLES_PER_CYCLE = round(scope.clock.adc_freq / scope.clock.clkgen_freq)
cycles = (np.arange(SAMPLES) - scope.adc.presamples) / SAMPLES_PER_CYCLE

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

## Inputs

The groups alternate: fixed, random, fixed, random, and so on. Every test uses the same inputs.

In [3]:
# 0 = fixed group (input FIXED_X), 1 = random group (a random 24-bit number)
groups = np.arange(N_TRACES) % 2
rng = np.random.default_rng(SEED)
inputs = np.where(groups == 0, FIXED_X, rng.integers(0, 2**24, N_TRACES))
warmup_inputs = rng.integers(0, 2**24, N_WARMUP)

## Functions

- `build_and_load(test)` builds the board code for one test and loads it onto the board.
- `warm_up()` runs the reduction `N_WARMUP` times so the board reaches a steady temperature. It runs once, in test 1.
- `record()` records one trace for every input.
- `welch_all` runs the Welch t-test on all traces. It is used in test 1 only.
- `welch_halves` runs the Welch t-test on each half of the traces. A sample leaks only if both halves cross `THRESHOLD` with the same sign.

In [4]:
# Runs make in the firmware folder. Stops with make's error output if it fails.
def make(*args):
    result = subprocess.run(["make", "-C", FIRMWARE, *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"make {' '.join(args)} failed:\n{result.stderr}")


# Builds the board code with trigger low at point `test` and loads it onto the board.
def build_and_load(test):
    make("clean")
    make(f"TRIGGER_STAGE={test}")
    cw.program_target(scope, cw.programmers.SAM4SProgrammer, HEX)

In [5]:
# Sends x to the board, runs one reduction, and returns the recorded power.
# Stops with an error if nothing is recorded or the board gives a wrong answer.
def capture(x):
    scope.arm()
    target.simpleserial_write("b", bytearray(x.to_bytes(3, "little")))
    if scope.capture():
        raise RuntimeError(f"x={x}: scope timed out, no trace recorded")
    reply = target.simpleserial_read("r", 4)
    if reply is None or int.from_bytes(reply, "little") != x % N:
        raise RuntimeError(f"x={x}: board sent {reply}, expected {x % N}")
    return scope.get_last_trace()


# Runs the reduction on the warm-up inputs and throws the traces away.
def warm_up():
    for x in tqdm(warmup_inputs, desc="warm-up"):
        capture(int(x))


# Records one trace per input. Returns the traces and the cycle of trigger low.
def record():
    # Record one trace to read the trigger length; the recording must reach trigger low
    capture(FIXED_X)
    if scope.adc.presamples + scope.adc.trig_count > SAMPLES:
        raise ValueError(f"set SAMPLES to at least {scope.adc.presamples + scope.adc.trig_count}")
    trigger_low = scope.adc.trig_count / SAMPLES_PER_CYCLE
    print(f"trigger low at cycle {trigger_low:.0f}")

    traces = np.array([capture(int(x)) for x in tqdm(inputs, desc="capture")])
    return traces, trigger_low

In [6]:
# Returns the Welch t-value at each sample between the traces in a and in b.
def welch_t(a, b):
    return ttest_ind(a, b, axis=0, equal_var=False).statistic


# Plots each t-value curve in curves ({label: t}) against the cycles, with lines
# at +THRESHOLD and -THRESHOLD and a vertical line at trigger low.
def plot_t(curves, trigger_low):
    xlabel = "cycles from trigger high"
    bound = np.full(SAMPLES, THRESHOLD)
    figure = cw.plot((cycles, bound), xlabel, "t") * cw.plot((cycles, -bound), xlabel, "t")
    for label, t in curves.items():
        figure *= cw.plot((cycles, t), xlabel, "t", label=label)
    return figure * hv.VLine(trigger_low)


# Saves a PNG copy of a plot as FIGURES/<name>.png.
# Returns the plot unchanged, so the cell still shows the interactive version.
def save_png(figure, name):
    os.makedirs(FIGURES, exist_ok=True)
    wide = figure.opts(hv.opts.Curve(linewidth=0.8, aspect=2, fig_inches=10, backend="matplotlib"),
                       hv.opts.Overlay(aspect=2, fig_inches=10, backend="matplotlib"), clone=True)
    hv.save(wide, os.path.join(FIGURES, name + ".png"), backend="matplotlib", dpi=100)
    return figure

In [7]:
# Runs the Welch t-test on all traces, prints the largest |t|,
# plots it, and saves a PNG copy as FIGURES/<name>.png.
def welch_all(traces, trigger_low, name):
    t = welch_t(traces[groups == 0], traces[groups == 1])
    print(f"max |t|: {np.max(np.abs(t)):.1f}")
    return save_png(plot_t({"all traces": t}, trigger_low), name)


# Runs the Welch t-test on each half of the traces, prints the leaking cycles,
# plots both halves, and saves a PNG copy as FIGURES/<name>.png.
def welch_halves(traces, trigger_low, name):
    fixed = traces[groups == 0]
    random = traces[groups == 1]
    half = len(fixed) // 2
    t1 = welch_t(fixed[:half], random[:half])
    t2 = welch_t(fixed[half:], random[half:])

    # A sample leaks when |t| is above the threshold in both halves, with the same sign
    leak = (np.abs(t1) > THRESHOLD) & (np.abs(t2) > THRESHOLD) & (np.sign(t1) == np.sign(t2))
    leak_cycles = np.unique(np.floor(cycles[leak]).astype(int))
    print("leaking cycles before trigger low:", leak_cycles[leak_cycles < trigger_low].tolist())
    print("leaking cycles after trigger low:", leak_cycles[leak_cycles >= trigger_low].tolist())
    return save_png(plot_t({"half 1": t1, "half 2": t2}, trigger_low), name)

## Test 1: trigger low after `str r2, [r5]`

In `barrett.S`, the pin-low store comes right after `str r2, [r5]` (writes the result to `r_out`).

In [8]:
build_and_load(1)

In [9]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	43db      	mvns	r3, r3
  400a96:	0fdb      	lsrs	r3, r3, #31
  400a98:	425b      	negs	r3, r3
  4

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`, `mvns r3, r3`, `lsrs r3, r3, #31`, `negs r3, r3`, `ands r3, r1`, `subs r2, r2, r3`, `str r2, [r5]`.

In [10]:
warm_up()

warm-up:   0%|          | 0/1000 [00:00<?, ?it/s]

In [11]:
traces, trigger_low = record()

trigger low at cycle 21


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [12]:
welch_all(traces, trigger_low, "test1_welch")

max |t|: 44.1


:Overlay
   .Curve.I          :Curve   [cycles from trigger high]   (t)
   .Curve.II         :Curve   [cycles from trigger high]   (t)
   .Curve.All_traces :Curve   [cycles from trigger high]   (t)
   .VLine.I          :VLine   [x,y]

![Welch t-test on all traces, test 1](figures/asm-sweep/test1_welch.png)

In [13]:
welch_halves(traces, trigger_low, "test1_halves")

leaking cycles before trigger low: [3, 7, 9, 11, 12, 14, 15, 16, 17, 18, 19]
leaking cycles after trigger low: [22, 35, 36, 37, 38, 39, 40, 41, 43, 59, 64, 118, 120, 121, 127, 128, 129, 137, 146, 149, 153, 156, 157, 177, 197, 217, 237]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 1](figures/asm-sweep/test1_halves.png)

## Test 2: trigger low after `subs r2, r2, r3`

In `barrett.S`, the pin-low store comes right after `subs r2, r2, r3` (r = r - sel). `str r2, [r5]` leaves the window.

In [14]:
build_and_load(2)

In [15]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	43db      	mvns	r3, r3
  400a96:	0fdb      	lsrs	r3, r3, #31
  400a98:	425b      	negs	r3, r3
  4

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`, `mvns r3, r3`, `lsrs r3, r3, #31`, `negs r3, r3`, `ands r3, r1`, `subs r2, r2, r3`.

In [16]:
traces, trigger_low = record()

trigger low at cycle 19


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [17]:
welch_halves(traces, trigger_low, "test2_halves")

leaking cycles before trigger low: [3, 7, 9, 11, 12, 14, 15, 16, 17]
leaking cycles after trigger low: [20, 21, 22, 35, 36, 37, 38, 39, 40, 41, 43, 59, 118, 120, 121, 126, 127, 128, 129, 137, 157, 177, 197, 217, 237]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 2](figures/asm-sweep/test2_halves.png)

## Test 3: trigger low after `ands r3, r1`

In `barrett.S`, the pin-low store comes right after `ands r3, r1` (sel = mask & N). `subs r2, r2, r3` leaves the window.

In [18]:
build_and_load(3)

In [19]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	43db      	mvns	r3, r3
  400a96:	0fdb      	lsrs	r3, r3, #31
  400a98:	425b      	negs	r3, r3
  4

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`, `mvns r3, r3`, `lsrs r3, r3, #31`, `negs r3, r3`, `ands r3, r1`.

In [20]:
traces, trigger_low = record()

trigger low at cycle 18


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [21]:
welch_halves(traces, trigger_low, "test3_halves")

leaking cycles before trigger low: [3, 7, 9, 11, 12, 13, 17]
leaking cycles after trigger low: [18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 41, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 136, 145, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 3](figures/asm-sweep/test3_halves.png)

## Test 4: trigger low after `negs r3, r3`

In `barrett.S`, the pin-low store comes right after `negs r3, r3` (mask = -reduce_flag). `ands r3, r1` leaves the window.

In [22]:
build_and_load(4)

In [23]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	43db      	mvns	r3, r3
  400a96:	0fdb      	lsrs	r3, r3, #31
  400a98:	425b      	negs	r3, r3
  4

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`, `mvns r3, r3`, `lsrs r3, r3, #31`, `negs r3, r3`.

In [24]:
traces, trigger_low = record()

trigger low at cycle 17


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [25]:
welch_halves(traces, trigger_low, "test4_halves")

leaking cycles before trigger low: [3, 7, 9, 11, 12, 14]
leaking cycles after trigger low: [17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 136, 145, 148, 154, 155, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 4](figures/asm-sweep/test4_halves.png)

## Test 5: trigger low after `lsrs r3, r3, #31`

In `barrett.S`, the pin-low store comes right after `lsrs r3, r3, #31` (reduce_flag = (~t) >> 31). `negs r3, r3` leaves the window.

In [26]:
build_and_load(5)

In [27]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	43db      	mvns	r3, r3
  400a96:	0fdb      	lsrs	r3, r3, #31
  400a98:	6037      	str	r7, [r6, #0

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`, `mvns r3, r3`, `lsrs r3, r3, #31`.

In [28]:
traces, trigger_low = record()

trigger low at cycle 16


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [29]:
welch_halves(traces, trigger_low, "test5_halves")

leaking cycles before trigger low: [3, 7, 9, 11, 12, 13, 14]
leaking cycles after trigger low: [17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 136, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 5](figures/asm-sweep/test5_halves.png)

## Test 6: trigger low after `mvns r3, r3`

In `barrett.S`, the pin-low store comes right after `mvns r3, r3` (~t). `lsrs r3, r3, #31` leaves the window.

In [30]:
build_and_load(6)

In [31]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	43db      	mvns	r3, r3
  400a96:	6037      	str	r7, [r6, #0]
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`, `mvns r3, r3`.

In [32]:
traces, trigger_low = record()

trigger low at cycle 15


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [33]:
welch_halves(traces, trigger_low, "test6_halves")

leaking cycles before trigger low: [3, 7, 9, 11, 12, 14]
leaking cycles after trigger low: [15, 17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 136, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 6](figures/asm-sweep/test6_halves.png)

## Test 7: trigger low after `subs r3, #5`

In `barrett.S`, the pin-low store comes right after `subs r3, #5` (t = r - N, second step). `mvns r3, r3` leaves the window.

In [34]:
build_and_load(7)

In [35]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	3b05      	subs	r3, #5
  400a94:	6037      	str	r7, [r6, #0]
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`, `subs r3, #5`.

In [36]:
traces, trigger_low = record()

trigger low at cycle 14


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [37]:
welch_halves(traces, trigger_low, "test7_halves")

leaking cycles before trigger low: [3, 7, 9, 10, 11, 12, 13]
leaking cycles after trigger low: [15, 17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 132, 136, 145, 148, 155, 156, 157, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 7](figures/asm-sweep/test7_halves.png)

## Test 8: trigger low after `sub.w r3, r2, #17664`

In `barrett.S`, the pin-low store comes right after `sub.w r3, r2, #17664` (t = r - N, first step). `subs r3, #5` leaves the window.

In [38]:
build_and_load(8)

In [39]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a92:	6037      	str	r7, [r6, #0]
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`, `sub.w r3, r2, #17664`.

In [40]:
traces, trigger_low = record()

trigger low at cycle 13


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [41]:
welch_halves(traces, trigger_low, "test8_halves")

leaking cycles before trigger low: [3, 7, 9]
leaking cycles after trigger low: [13, 17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 119, 120, 125, 126, 127, 128, 136, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 8](figures/asm-sweep/test8_halves.png)

## Test 9: trigger low after `mls r2, r1, r3, r2`

In `barrett.S`, the pin-low store comes right after `mls r2, r1, r3, r2` (r = x - q * N). `sub.w r3, r2, #17664` leaves the window.

In [42]:
build_and_load(9)

In [43]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	fb01 2213 	mls	r2, r1, r3, r2
  400a8e:	6037      	str	r7, [r6, #0]
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`, `mls r2, r1, r3, r2`.

In [44]:
traces, trigger_low = record()

trigger low at cycle 12


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [45]:
welch_halves(traces, trigger_low, "test9_halves")

leaking cycles before trigger low: [3, 7, 9, 10]
leaking cycles after trigger low: [12, 13, 15, 16, 17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 136, 148, 155, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 9](figures/asm-sweep/test9_halves.png)

## Test 10: trigger low after `movw r1, #17669`

In `barrett.S`, the pin-low store comes right after `movw r1, #17669` (loads N). `mls r2, r1, r3, r2` leaves the window.

In [46]:
build_and_load(10)

In [47]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8a:	6037      	str	r7, [r6, #0]
  400a8c:	fb01 2213 	mls	r2, r1, r3, r2
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`, `movw r1, #17669`.

In [48]:
traces, trigger_low = record()

trigger low at cycle 10


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [49]:
welch_halves(traces, trigger_low, "test10_halves")

leaking cycles before trigger low: [3, 7, 8, 9]
leaking cycles after trigger low: [11, 12, 14, 16, 17, 18, 19, 20, 33, 34, 35, 36, 37, 38, 39, 41, 57, 116, 117, 118, 119, 124, 125, 126, 127, 135, 154, 155, 175, 184, 195, 215, 235]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 10](figures/asm-sweep/test10_halves.png)

## Test 11: trigger low after `umull r1, r3, r3, r2`

In `barrett.S`, the pin-low store comes right after `umull r1, r3, r3, r2` (q = (x * mu) >> 32). `movw r1, #17669` leaves the window.

In [50]:
build_and_load(11)

In [51]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	fba3 1302 	umull	r1, r3, r3, r2
  400a86:	6037      	str	r7, [r6, #0]
  400a88:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8c:	fb01 2213 	mls	r2, r1, r3, r2
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`, `umull r1, r3, r3, r2`.

In [52]:
traces, trigger_low = record()

trigger low at cycle 9


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [53]:
welch_halves(traces, trigger_low, "test11_halves")

leaking cycles before trigger low: [3, 7]
leaking cycles after trigger low: [9, 11, 12, 14, 16, 17, 18, 19, 20, 33, 34, 35, 36, 37, 38, 39, 41, 57, 116, 117, 118, 119, 124, 125, 126, 127, 135, 155, 175, 195, 215, 235]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 11](figures/asm-sweep/test11_halves.png)

## Test 12: trigger low after `ldr r5, =r_out`

In `barrett.S`, the pin-low store comes right after `ldr r5, =r_out` (loads the address of `r_out`). `umull r1, r3, r3, r2` leaves the window.

In [54]:
build_and_load(12)

In [55]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a82:	6037      	str	r7, [r6, #0]
  400a84:	fba3 1302 	umull	r1, r3, r3, r2
  400a88:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8c:	fb01 2213 	mls	r2, r1, r3, r2
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`, `ldr r5, =r_out`.

In [56]:
traces, trigger_low = record()

trigger low at cycle 8


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [57]:
welch_halves(traces, trigger_low, "test12_halves")

leaking cycles before trigger low: [3, 7]
leaking cycles after trigger low: [9, 11, 12, 14, 15, 16, 17, 18, 19, 20, 33, 34, 35, 36, 37, 38, 39, 41, 57, 64, 116, 117, 118, 119, 124, 125, 126, 127, 135, 155, 175, 195, 215, 235]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 12](figures/asm-sweep/test12_halves.png)

## Test 13: trigger low after `ldr r3, =PARAM_N_MU`

In `barrett.S`, the pin-low store comes right after `ldr r3, =PARAM_N_MU` (loads mu). `ldr r5, =r_out` leaves the window.

In [58]:
build_and_load(13)

In [59]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	4b0c      	ldr	r3, [pc, #48]	@ (400ab0 <barrett_window+0x40>)
  400a80:	6037      	str	r7, [r6, #0]
  400a82:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a84:	fba3 1302 	umull	r1, r3, r3, r2
  400a88:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8c:	fb01 2213 	mls	r2, r1, r3, r2
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`, `ldr r3, =PARAM_N_MU`.

In [60]:
traces, trigger_low = record()

trigger low at cycle 7


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [61]:
welch_halves(traces, trigger_low, "test13_halves")

leaking cycles before trigger low: [3]
leaking cycles after trigger low: [10, 12, 14, 15, 18, 19, 20, 21, 22, 23, 36, 37, 38, 39, 40, 41, 42, 44, 60, 118, 119, 120, 121, 122, 127, 128, 129, 130, 134, 138, 158, 178, 198, 218, 238]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 13](figures/asm-sweep/test13_halves.png)

## Test 14: trigger low after `ldr r2, [r5]`

In `barrett.S`, the pin-low store comes right after `ldr r2, [r5]` (reads x). `ldr r3, =PARAM_N_MU` leaves the window.

In [62]:
build_and_load(14)

In [63]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	682a      	ldr	r2, [r5, #0]
  400a7e:	6037      	str	r7, [r6, #0]
  400a80:	4b0b      	ldr	r3, [pc, #44]	@ (400ab0 <barrett_window+0x40>)
  400a82:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a84:	fba3 1302 	umull	r1, r3, r3, r2
  400a88:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8c:	fb01 2213 	mls	r2, r1, r3, r2
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): `ldr r2, [r5]`.

In [64]:
traces, trigger_low = record()

trigger low at cycle 4


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [65]:
welch_halves(traces, trigger_low, "test14_halves")

leaking cycles before trigger low: []
leaking cycles after trigger low: [8, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 117, 118, 119, 120, 125, 126, 127, 128, 136, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 14](figures/asm-sweep/test14_halves.png)

## Test 15: trigger low right after trigger high

In `barrett.S`, the pin-low store comes right after the pin-high store, so the window is empty. `ldr r2, [r5]` leaves the window.

In [66]:
build_and_load(15)

In [67]:
# Print the board instructions of barrett_window
!arm-none-eabi-objdump --disassemble=barrett_window {ELF}


simpleserial-barrett-asm/simpleserial-barrett-asm-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400a70 <barrett_window>:
  400a70:	b5f0      	push	{r4, r5, r6, r7, lr}
  400a72:	4c0c      	ldr	r4, [pc, #48]	@ (400aa4 <barrett_window+0x34>)
  400a74:	4e0c      	ldr	r6, [pc, #48]	@ (400aa8 <barrett_window+0x38>)
  400a76:	2780      	movs	r7, #128	@ 0x80
  400a78:	4d0c      	ldr	r5, [pc, #48]	@ (400aac <barrett_window+0x3c>)
  400a7a:	6027      	str	r7, [r4, #0]
  400a7c:	6037      	str	r7, [r6, #0]
  400a7e:	682a      	ldr	r2, [r5, #0]
  400a80:	4b0b      	ldr	r3, [pc, #44]	@ (400ab0 <barrett_window+0x40>)
  400a82:	4d0c      	ldr	r5, [pc, #48]	@ (400ab4 <barrett_window+0x44>)
  400a84:	fba3 1302 	umull	r1, r3, r3, r2
  400a88:	f244 5105 	movw	r1, #17669	@ 0x4505
  400a8c:	fb01 2213 	mls	r2, r1, r3, r2
  400a90:	f5a2 438a 	sub.w	r3, r2, #17664	@ 0x4500
  400a94:	3b05      	subs	r3, #5
  400a96:	43db      	mvns	r3, r3
  400a98:	0fdb      	lsrs	r3, r3, #3

Trigger window (between `str r7, [r4]` and `str r7, [r6]`): no instructions.

In [68]:
traces, trigger_low = record()

trigger low at cycle 2


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [69]:
welch_halves(traces, trigger_low, "test15_halves")

leaking cycles before trigger low: []
leaking cycles after trigger low: [5, 8, 10, 12, 13, 15, 17, 18, 19, 20, 21, 34, 35, 36, 37, 38, 39, 40, 42, 58, 59, 117, 118, 119, 120, 125, 126, 127, 128, 136, 156, 176, 196, 216, 236]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 15](figures/asm-sweep/test15_halves.png)

## Disconnect

Releases the Husky so other notebooks and programs can use it.

In [70]:
scope.dis()
target.dis()